# Lab 1 — Approval Gates

**Goal:** Build the fundamental HITL pattern: an agent that pauses before each consequential action and waits for human sign-off.

**Why this matters:** Every enterprise AI system needs this. Without approval gates, a bug in your agent logic can send 10,000 emails, delete production data, or execute large financial trades before anyone notices.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from dotenv import load_dotenv
load_dotenv(override=True)
from hitl import Step, ApprovalGate, ApprovalResult, Decision, HITLAgent
from openai import OpenAI

client = OpenAI()
print('HITL framework loaded ✓')

## 1. Auto-approve mode (for testing)
Always use this in automated tests — never prompt in notebooks unless you intend to interact.

In [ ]:
steps = [
    Step('research', 'List 3 benefits of solar energy.', confidence=0.9),
    Step('draft',    'Write a 100-word intro about solar energy.', confidence=0.8),
    Step('email',    'Send the draft to subscribers@example.com.', confidence=0.5, cost_estimate_usd=0.01),
]

def simple_executor(step: Step, feedback_ctx: str) -> str:
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': step.description}],
        max_tokens=150,
    )
    return response.choices[0].message.content

gate = ApprovalGate(backend='auto')  # auto-approves everything
agent = HITLAgent(gate=gate)
results = agent.run(steps, simple_executor)

for r in results:
    print(f"[{r['step']}] {r['decision']}")
    print(r['result'][:200])
    print()

## 2. Threshold-based auto-escalation
Steps with low confidence or high cost automatically require approval.

In [ ]:
# Simulate what needs_human() returns for each step
gate_with_thresholds = ApprovalGate(
    backend='auto',
    confidence_threshold=0.75,
    cost_threshold_usd=0.005,
)

for step in steps:
    needs = gate_with_thresholds.needs_human(step)
    reason = []
    if step.confidence < 0.75:
        reason.append(f'confidence {step.confidence:.0%} < 75%')
    if step.cost_estimate_usd > 0.005:
        reason.append(f'cost ${step.cost_estimate_usd:.3f} > $0.005')
    print(f"{step.name:12} needs_human={needs}  {'(' + ', '.join(reason) + ')' if reason else '(auto-approved)'}")

## 3. Callback gate — simulating a UI
In production, the gate calls your Gradio UI, Slack bot, or email system.  
Here we simulate it with a callback that always approves.

In [ ]:
approval_log = []

def my_callback(step: Step) -> ApprovalResult:
    """Simulated UI callback — in reality this would open a browser modal."""
    approval_log.append(f'UI presented step: {step.name} (confidence={step.confidence:.0%})')
    # Simulate: reject the 'email' step
    if step.name == 'email':
        return ApprovalResult(Decision.REJECT, feedback='Not ready to send yet')
    return ApprovalResult(Decision.APPROVE)

gate_cb = ApprovalGate(backend='callback', callback=my_callback, confidence_threshold=0.6)
agent_cb = HITLAgent(gate=gate_cb)
results_cb = agent_cb.run(steps, simple_executor)

print('\nApproval log:')
for entry in approval_log:
    print(' ', entry)

print('\nResults:')
for r in results_cb:
    print(f"  [{r['step']}] {r['decision']}: {str(r['result'])[:100]}")

## 4. Halting on rejection
Notice that when 'email' was rejected, the workflow stopped. Steps after a rejection never execute.

In [ ]:
executed = [r['step'] for r in results_cb if r['result'] and 'SKIPPED' not in str(r['result'])]
skipped  = [r['step'] for r in results_cb if r['result'] and 'SKIPPED' in str(r['result'])]
print('Executed:', executed)
print('Skipped: ', skipped)

## Exercise
Extend the callback to implement a **budget gate**: automatically reject any step whose `cost_estimate_usd` would push the total run cost above $0.05.  
Track a running total and reject when it would be exceeded.

In [ ]:
# Your code here
